In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, learning_curve
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import OLSInfluence


# 1. Data Cleaning: Handle missing values, drop columns, and clean up the dataset
def clean_data(df):
    date_columns = ['Date_birth', 'Date_start_contract', 'Date_last_renewal', 
                    'Date_next_renewal', 'Date_driving_licence', 'Date_lapse']
    df[date_columns] = df[date_columns].apply(pd.to_datetime, dayfirst=True, errors='coerce')
    drop_columns = ['Date_start_contract', 'Date_next_renewal', 
                    'Date_lapse', 'Length', 'Policies_in_force', 'Max_policies', 'Max_products']
    df = df.drop(columns=drop_columns)
    df['Distribution_channel'] = pd.to_numeric(
        df['Distribution_channel'].replace('00/01/1900', np.nan), errors='coerce'
    )
    df = df.dropna(subset=['Distribution_channel'])
    return df

# 2. Preprocessing: Filter data, apply conditions, and preprocess the columns
def preprocess_data(df):
    df = df[df['Type_risk'] == 3.0]
    df = df[(df["N_claims_year"] >= 0) & (df["N_claims_year"] <= 3)]
    return df

# 3. Feature Engineering: Create new features from existing data
def feature_engineering(df):
    # Create derived features (Driving experience and Age in 2018)
    df['Driving_experience_years'] = df['Date_last_renewal'].dt.year - df['Date_driving_licence'].dt.year
    df['Age_in_2018'] = df['Date_last_renewal'].dt.year - df['Date_birth'].dt.year
    # Group 'Age_in_2018' into age bins '16-24', '25-39', '40-64', '65+'
    age_bins = [16, 24, 39, 64, float('inf')]
    age_labels = ['16-24', '25-39', '40-64', '65+']
    df['Age_Group'] = pd.cut(df['Age_in_2018'], bins=age_bins, labels=age_labels, right=True)
    df = df.drop(columns=['Age_in_2018', 'Date_driving_licence', 'Date_birth', 'Date_last_renewal'])
    return df

# 4. One-hot Encoding using OneHotEncoder from sklearn
def one_hot_encode(df, categorical_cols):
    encoder = OneHotEncoder(sparse_output=False)
    encoded_cols = encoder.fit_transform(df[categorical_cols])
    encoded_df = pd.DataFrame(encoded_cols, columns=encoder.get_feature_names_out(categorical_cols))
    encoded_df.index = df.index
    df = df.drop(columns=categorical_cols).join(encoded_df)
    return df

# Main function to process motor data and evaluate models
def process_motor_data(file_path):
    df_motor = pd.read_csv(file_path, delimiter=';')
    df_motor = clean_data(df_motor)
    df_motor = preprocess_data(df_motor)
    df_motor = feature_engineering(df_motor)
    categorical_columns = ['Age_Group', 'Type_fuel']
    df_motor = one_hot_encode(df_motor, categorical_columns)
    return df_motor

df_processed= process_motor_data('motor.csv')



C:\Users\Prapatsorn\AppData\Local\Temp\ipykernel_32500\2302338769.py:57: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_motor = pd.read_csv(file_path, delimiter=';')


In [2]:
df_processed["Cost_claims_year"].describe()

count     78298.000000
mean        140.907951
std        1604.563768
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max      260853.240000
Name: Cost_claims_year, dtype: float64

In [17]:
df_processed.head(10)

,ID,Distribution_channel,Seniority,Lapse,Payment,Premium,Cost_claims_year,N_claims_year,N_claims_history,R_Claims_history,...,Value_vehicle,N_doors,Weight,Driving_experience_years,Age_Group_16-24,Age_Group_25-39,Age_Group_40-64,Age_Group_65+,Type_fuel_D,Type_fuel_P
6,3,0.0,15,0,0,380.20,0.00,0,0,0.00,...,16030.0,5,1105,20,0.0,0.0,1.0,0.0,0.0,1.0
7,3,0.0,15,0,0,393.50,0.00,0,0,0.00,...,16030.0,5,1105,21,0.0,0.0,1.0,0.0,0.0,1.0
8,3,0.0,15,0,0,393.50,0.00,0,0,0.00,...,16030.0,5,1105,22,0.0,0.0,1.0,0.0,0.0,1.0
9,3,0.0,15,0,0,395.47,0.00,0,0,0.00,...,16030.0,5,1105,23,0.0,0.0,1.0,0.0,0.0,1.0
15,6,0.0,13,0,0,250.52,0.00,0,2,0.16,...,9927.0,4,1055,26,0.0,0.0,1.0,0.0,0.0,1.0
16,6,0.0,13,0,0,258.04,0.00,0,2,0.16,...,9927.0,4,1055,27,0.0,0.0,1.0,0.0,0.0,1.0
17,6,0.0,13,0,0,259.32,0.00,0,2,0.16,...,9927.0,4,1055,28,0.0,0.0,1.0,0.0,0.0,1.0
18,7,1.0,3,0,0,340.43,2162.89,3,4,2.00,...,16550.0,5,1168,16,0.0,0.0,0.0,1.0,0.0,1.0
19,8,0.0,18,0,0,335.82,0.00,0,6,0.34,...,27330.0,4,1310,54,0.0,0.0,0.0,1.0,0.0,1.0
20,8,0.0,18,0,0,345.91,0.00,0,6,0.34,...,27330.0,4,1310,55,0.0,0.0,0.0,1.0,0.0,1.0


In [9]:
df_processed.describe()

,ID,Distribution_channel,Seniority,Lapse,Payment,Premium,Cost_claims_year,N_claims_year,N_claims_history,R_Claims_history,...,Value_vehicle,N_doors,Weight,Driving_experience_years,Age_Group_16-24,Age_Group_25-39,Age_Group_40-64,Age_Group_65+,Type_fuel_D,Type_fuel_P
count,78298.000000,78298.000000,78298.000000,78298.000000,78298.000000,78298.000000,78298.000000,78298.000000,78298.000000,78298.000000,...,78298.000000,78298.000000,78298.000000,78298.000000,78298.000000,78298.000000,78298.000000,78298.000000,78298.000000,78298.000000
mean,27054.441007,0.463945,6.567396,0.219112,0.348004,333.589485,140.907951,0.272957,2.577844,0.404023,...,19458.843788,4.468863,1228.280582,23.954226,0.018289,0.316598,0.567115,0.097997,0.622519,0.377481
std,15282.287970,0.498702,6.203594,0.451697,0.476340,136.427371,1604.563768,0.669455,3.519084,0.662750,...,8471.995321,0.843961,225.888448,12.400218,0.133996,0.465152,0.495478,0.297313,0.484760,0.484760
min,3.000000,0.000000,1.000000,0.000000,0.000000,40.710000,0.000000,0.000000,0.000000,0.000000,...,270.460000,2.000000,246.000000,-3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,13909.000000,0.000000,3.000000,0.000000,0.000000,253.772500,0.000000,0.000000,0.000000,0.000000,...,14050.000000,4.000000,1070.000000,14.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27339.000000,0.000000,4.000000,0.000000,0.000000,300.035000,0.000000,0.000000,1.000000,0.110000,...,18100.000000,5.000000,1205.000000,23.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000
75%,40397.000000,1.000000,8.000000,0.000000,1.000000,370.767500,0.000000,0.000000,4.000000,0.560000,...,22830.000000,5.000000,1357.000000,33.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000
max,53502.000000,1.000000,40.000000,6.000000,1.000000,2993.340000,260853.240000,3.000000,50.000000,26.070000,...,220675.800000,6.000000,2710.000000,74.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


#### check average frequency and severity 

In [3]:
# Check average frequency
total_number_claim = df_processed[df_processed['N_claims_year'] > 0]['N_claims_year'].sum()
print(f"Total number of claims: {total_number_claim}")
total_policy_year = df_processed["ID"].count()
print(f"Total policy years: {total_policy_year}")
avg_frequency = total_number_claim / total_policy_year
print(f"Average frequency: {avg_frequency}")

# Check average severity
total_claim_costs = df_processed['Cost_claims_year'].sum()
print(f"Total claim costs: {total_claim_costs}")
avg_severity = total_claim_costs / total_number_claim
print(f"Average severity: {avg_severity}")

#average total costs 
average_total_costs = avg_frequency * avg_severity 
print(f"Average total costs: {average_total_costs}")



Total number of claims: 21372
Total policy years: 78298
Average frequency: 0.2729571636567984
Total claim costs: 11032810.73
Average severity: 516.2273409133446
Average total costs: 140.90795077779765


In [4]:
import numpy as np

# Filter to include only policies with claims
claims_data = df_processed[df_processed['N_claims_year'] > 0]

# Calculate variance and standard deviation for frequency
variance_frequency = np.sum((claims_data['N_claims_year'] - avg_frequency) ** 2) / total_policy_year
std_dev_frequency = np.sqrt(variance_frequency)

print(f"Standard Deviation of frequency: {std_dev_frequency}")


Standard Deviation of frequency: 0.6217218042501146


### Split Training and testing set

In [13]:
from sklearn.model_selection import train_test_split

# Splitting the data into 80% training and 20% testing
def split_and_export_data(df):
    X = df.drop(columns=['Cost_claims_year'])
    y = df['Cost_claims_year']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Convert training and testing sets to DataFrames for easier export
    train_set = X_train.copy()
    train_set['Cost_claims_year'] = y_train.values  # Add target column back to training set
    test_set = X_test.copy()
    test_set['Cost_claims_year'] = y_test.values    # Add target column back to testing set

    # Export to CSV
    train_set.to_csv('train_set.csv', index=False)
    test_set.to_csv('test_set.csv', index=False)

    return train_set, test_set

# Example usage with df_processed
train_set, test_set = split_and_export_data(df_processed)


In [1]:
import pandas as pd 
train_set = pd.read_csv('train_set.csv')
test_set = pd.read_csv('test_set.csv')

In [2]:
train_set.describe()

,ID,Distribution_channel,Seniority,Lapse,Payment,Premium,N_claims_year,N_claims_history,R_Claims_history,Type_risk,...,N_doors,Weight,Driving_experience_years,Age_Group_16-24,Age_Group_25-39,Age_Group_40-64,Age_Group_65+,Type_fuel_D,Type_fuel_P,Cost_claims_year
count,62638.000000,62638.000000,62638.000000,62638.000000,62638.000000,62638.000000,62638.000000,62638.000000,62638.000000,62638.0,...,62638.000000,62638.000000,62638.000000,62638.000000,62638.000000,62638.000000,62638.000000,62638.000000,62638.000000,62638.000000
mean,27043.408378,0.464542,6.556052,0.219819,0.347888,333.810682,0.274721,2.581596,0.404951,3.0,...,4.470593,1228.544095,23.968518,0.018663,0.316469,0.566429,0.098439,0.622258,0.377742,140.752655
std,15299.786351,0.498745,6.202948,0.451988,0.476304,136.285321,0.671890,3.528229,0.662597,0.0,...,0.842341,225.916251,12.418779,0.135332,0.465102,0.495571,0.297909,0.484826,0.484826,1659.854800
min,3.000000,0.000000,1.000000,0.000000,0.000000,40.710000,0.000000,0.000000,0.000000,3.0,...,2.000000,246.000000,-3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,13855.250000,0.000000,3.000000,0.000000,0.000000,253.980000,0.000000,0.000000,0.000000,3.0,...,4.000000,1070.000000,14.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27339.500000,0.000000,4.000000,0.000000,0.000000,300.105000,0.000000,1.000000,0.110000,3.0,...,5.000000,1205.000000,23.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000
75%,40411.500000,1.000000,8.000000,0.000000,1.000000,370.740000,0.000000,4.000000,0.560000,3.0,...,5.000000,1357.000000,33.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000
max,53502.000000,1.000000,40.000000,6.000000,1.000000,2797.510000,3.000000,50.000000,21.470000,3.0,...,6.000000,2710.000000,74.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,260853.240000


In [4]:
test_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15660 entries, 0 to 15659
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        15660 non-null  int64  
 1   Distribution_channel      15660 non-null  float64
 2   Seniority                 15660 non-null  int64  
 3   Lapse                     15660 non-null  int64  
 4   Payment                   15660 non-null  int64  
 5   Premium                   15660 non-null  float64
 6   N_claims_year             15660 non-null  int64  
 7   N_claims_history          15660 non-null  int64  
 8   R_Claims_history          15660 non-null  float64
 9   Type_risk                 15660 non-null  int64  
 10  Area                      15660 non-null  int64  
 11  Second_driver             15660 non-null  int64  
 12  Year_matriculation        15660 non-null  int64  
 13  Power                     15660 non-null  int64  
 14  Cylind

In [5]:
test_set['Cost_claims_year'].describe()

count     15660.000000
mean        141.529113
std        1361.177805
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max      128809.730000
Name: Cost_claims_year, dtype: float64

#### check average frequency and severity in training set

In [20]:
# Check average frequency
total_number_claim_train = train_set[train_set['N_claims_year'] > 0]['N_claims_year'].sum()
print(f"Total number of claims: {total_number_claim}")
total_policy_year_train = train_set["ID"].count()
print(f"Total policy years: {total_policy_year_train}")
avg_frequency_train = total_number_claim_train / total_policy_year_train
print(f"Average frequency: {avg_frequency_train}")

# Check average severity
total_claim_costs_train = train_set['Cost_claims_year'].sum()
print(f"Total claim costs: {total_claim_costs_train}")
avg_severity_train = total_claim_costs_train / total_number_claim_train
print(f"Average severity: {avg_severity_train}")

#average total costs 
average_total_costs_train = avg_frequency_train * avg_severity_train 
print(f"Average total costs: {average_total_costs_train}")



Total number of claims: 21372
Total policy years: 62638
Average frequency: 0.27472141511542514
Total claim costs: 8816464.82
Average severity: 512.3468630869363
Average total costs: 140.75265525719212


#### check average frequency and severity in testing set

In [21]:
# Check average frequency
total_number_claim_test = test_set[test_set['N_claims_year'] > 0]['N_claims_year'].sum()
print(f"Total number of claims: {total_number_claim}")
total_policy_year_test = test_set["ID"].count()
print(f"Total policy years: {total_policy_year_test}")
avg_frequency_test = total_number_claim_test / total_policy_year_test
print(f"Average frequency: {avg_frequency_test}")

# Check average severity
total_claim_costs_test = test_set['Cost_claims_year'].sum()
print(f"Total claim costs: {total_claim_costs_test}")
avg_severity_test = total_claim_costs_test / total_number_claim_test
print(f"Average severity: {avg_severity_test}")

#average total costs 
average_total_costs_test = avg_frequency_test * avg_severity_test 
print(f"Average total costs: {average_total_costs_test}")

Total number of claims: 21372
Total policy years: 15660
Average frequency: 0.26590038314176245
Total claim costs: 2216345.91
Average severity: 532.2636671469741
Average total costs: 141.5291130268199
